# Tutorial: Feature Engineering

> **Type:** Tutorial. Follow along step by step. Your only job is to make it work. Don't skip cells.
>
> **Time:** ~1.5-2 hours
> **Prerequisites:** Module 07 (data cleaning), Module 09 (EDA), Module 13 (cross-validation)
> **What you'll build:** A leakage-safe `ColumnTransformer` pipeline that turns a raw e-commerce orders dataset into an ML-ready feature matrix, with encoded categoricals, scaled numerics, datetime features, interaction terms, and binned variables.

---

## What We're Building

You'll work with a synthetic e-commerce orders dataset. It has numeric columns (customer age, items in cart), low-cardinality categorical columns (region, payment method), a high-cardinality ID column (`customer_id`), and a datetime column (order timestamp). The target is order value. By the end, you'll have a single reusable `sklearn.pipeline.Pipeline` that takes raw rows in and produces a clean feature matrix out, fit only on training data.

Don't worry yet about understanding every line. That's what `explanation.md` is for. Here, just follow the steps and watch it work.

## Step 0: Setup

Install and import everything we need.

In [ ]:
# !pip install numpy pandas scikit-learn matplotlib category_encoders

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import (
    OneHotEncoder, OrdinalEncoder, StandardScaler, MinMaxScaler,
    RobustScaler, PolynomialFeatures, KBinsDiscretizer
)
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.model_selection import cross_val_score

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

plt.rcParams['figure.figsize'] = (10, 4)
plt.rcParams['font.size'] = 11
print('Setup complete.')

## Step 1: Build a Synthetic Orders Dataset

We simulate an e-commerce orders table, the kind of raw data you'd pull from a production database. It mixes numeric, low-cardinality categorical, high-cardinality categorical, and datetime columns. The target (`order_value`) depends on interactions between columns, so feature engineering has a visible payoff later.

In [ ]:
n = 4000

regions = np.random.choice(['North', 'South', 'East', 'West'], size=n, p=[0.3, 0.25, 0.25, 0.2])
customer_id = np.random.randint(1000, 1000 + 400, size=n)  # high-cardinality: 400 unique customers
payment_method = np.random.choice(['credit_card', 'paypal', 'gift_card', 'bank_transfer'], size=n)
customer_age = np.random.normal(40, 12, size=n).clip(18, 80)

start = pd.Timestamp('2024-01-01')
order_timestamp = start + pd.to_timedelta(np.random.randint(0, 365 * 24 * 60, size=n), unit='m')

items_in_cart = np.random.poisson(3, size=n) + 1

# Region and payment method genuinely shift spend: this is the signal we want features to expose
region_premium = pd.Series(regions).map({'North': 1.15, 'South': 0.95, 'East': 1.05, 'West': 0.90}).values
weekend_boost = pd.Series(order_timestamp).dt.dayofweek.isin([5, 6]).astype(int).values * 8

order_value = (
    15 * items_in_cart * region_premium
    + 0.3 * customer_age
    + weekend_boost
    + np.random.normal(0, 10, size=n)
).clip(5, None)

df = pd.DataFrame({
    'customer_id': customer_id,
    'region': regions,
    'payment_method': payment_method,
    'customer_age': customer_age,
    'items_in_cart': items_in_cart,
    'order_timestamp': order_timestamp,
    'order_value': order_value,
})

df.head()

> **Check:** you should see 4000 rows and 7 columns, with `order_timestamp` as a datetime and `region`/`payment_method` as strings. If `order_timestamp` shows as an object (string) column instead of a datetime, re-run the cell above.

## Step 2: Split Before You Engineer Anything

This is the most important step in the notebook. Every encoder and scaler we fit below must see **only the training set**. If you fit on the full dataset first, the encoding statistics and scaler means will have seen the test set. That's leakage, and it makes your validation metrics lie to you.

In [ ]:
X = df.drop(columns=['order_value'])
y = df['order_value']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_SEED)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")

## Step 3: Datetime Feature Extraction

`order_timestamp` on its own is useless to a model: it's just a large, arbitrary number. The signal is *inside* it: day of week, hour, whether it's a weekend. We also encode `hour` and `day_of_week` cyclically with sin/cos, so that hour 23 and hour 0 are treated as neighbours instead of being as far apart as possible.

In [ ]:
def add_datetime_features(frame):
    frame = frame.copy()
    ts = frame['order_timestamp']
    frame['hour'] = ts.dt.hour
    frame['day_of_week'] = ts.dt.dayofweek       # 0 = Monday
    frame['month'] = ts.dt.month
    frame['is_weekend'] = ts.dt.dayofweek.isin([5, 6]).astype(int)

    # Cyclical encoding: makes 23:00 and 00:00 numerically adjacent
    frame['hour_sin'] = np.sin(2 * np.pi * frame['hour'] / 24)
    frame['hour_cos'] = np.cos(2 * np.pi * frame['hour'] / 24)
    frame['dow_sin'] = np.sin(2 * np.pi * frame['day_of_week'] / 7)
    frame['dow_cos'] = np.cos(2 * np.pi * frame['day_of_week'] / 7)

    return frame.drop(columns=['order_timestamp'])

X_train_dt = add_datetime_features(X_train)
X_test_dt = add_datetime_features(X_test)
X_train_dt.head()

> **What just happened?** `order_timestamp` became eight numeric columns. `hour_sin` and `hour_cos` together place the hour on a circle. There's no leakage risk here, because this is a pure function of the timestamp, not of the target.

## Step 4: Visualise Why Cyclical Encoding Matters

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].scatter(range(24), range(24), c=range(24), cmap='twilight')
axes[0].set_title('Raw hour (0-23)\nhour 23 and hour 0 look maximally far apart')
axes[0].set_xlabel('hour')
axes[0].set_ylabel('hour')

hours = np.arange(24)
hs, hc = np.sin(2*np.pi*hours/24), np.cos(2*np.pi*hours/24)
axes[1].scatter(hs, hc, c=hours, cmap='twilight')
for h in [0, 6, 12, 18, 23]:
    axes[1].annotate(str(h), (hs[h], hc[h]))
axes[1].set_title('sin/cos encoded hour\nhour 23 and hour 0 are now neighbours')
axes[1].set_xlabel('hour_sin')
axes[1].set_ylabel('hour_cos')

plt.tight_layout()
plt.show()

## Step 5: Encoding Categorical Variables

We have categoricals with very different cardinality:

- `region`: 4 unique values → **one-hot encode**. Cheap, implies no ordering, and the model can learn a separate weight per region.
- `customer_id`: ~400 unique values. One-hot encoding would add 400 columns for one field. Instead we use **frequency encoding**, which is safe and doesn't leak. Step 11 covers target encoding with proper cross-validation, for cases where frequency isn't informative enough.

`payment_method` (4 values, no natural order) is also one-hot encoded.

In [ ]:
# Frequency encoding for high-cardinality customer_id: computed on TRAIN ONLY
customer_freq = X_train_dt['customer_id'].value_counts(normalize=True)

def add_frequency_encoding(frame, freq_map):
    frame = frame.copy()
    # unseen customers in test set get frequency 0 (never seen in training)
    frame['customer_freq'] = frame['customer_id'].map(freq_map).fillna(0.0)
    return frame.drop(columns=['customer_id'])

X_train_freq = add_frequency_encoding(X_train_dt, customer_freq)
X_test_freq = add_frequency_encoding(X_test_dt, customer_freq)  # reuses TRAIN frequencies

X_train_freq[['customer_freq']].describe()

> **What just happened?** `customer_freq` replaces 400 raw IDs with a single number: how often that customer appears in the *training* set. Notice that `X_test_freq` reused `customer_freq` computed from training. That's the leakage-safe pattern. A customer who only appears in the test set correctly gets frequency 0.

## Step 6: Interaction and Polynomial Features

`items_in_cart` and `region` interact: the same cart size is worth more in a premium region. We can't multiply a category directly. But after one-hot encoding we can create interaction terms, or let `PolynomialFeatures` generate all pairwise products of the numeric columns.

In [ ]:
numeric_cols = ['customer_age', 'items_in_cart']

poly = PolynomialFeatures(degree=2, include_bias=False, interaction_only=False)
poly_train = poly.fit_transform(X_train_freq[numeric_cols])
poly_feature_names = poly.get_feature_names_out(numeric_cols)

pd.DataFrame(poly_train, columns=poly_feature_names).head()

> **What just happened?** `PolynomialFeatures(degree=2)` turned 2 columns into 5: the originals, their squares, and their product (`customer_age items_in_cart`). For a linear model, that product term is the only way to capture "older customers with big carts spend disproportionately more". A tree-based model would find this split on its own, but a linear model needs it spelled out.

## Step 7: Scaling Numeric Features

Scaling matters for linear and distance-based models, not for trees. We compare three scalers on `customer_age`:

- `StandardScaler`: mean 0, standard deviation 1. The default choice.
- `MinMaxScaler`: squashes values into [0, 1]. Use it when you need bounded input (for example, neural nets).
- `RobustScaler`: uses the median and IQR. Use it when outliers would distort the mean and standard deviation.

In [ ]:
age = X_train_freq[['customer_age']]

scalers = {
    'StandardScaler': StandardScaler(),
    'MinMaxScaler': MinMaxScaler(),
    'RobustScaler': RobustScaler(),
}

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, (name, scaler) in zip(axes, scalers.items()):
    scaled = scaler.fit_transform(age)
    ax.hist(scaled, bins=30, color='#4E79A7')
    ax.set_title(name)
plt.tight_layout()
plt.show()

## Step 8: Binning a Continuous Variable

Sometimes the *bucket* a value falls into matters more than its exact value. Marketing teams segment customers into age bands rather than modelling age continuously. `KBinsDiscretizer` automates this.

In [ ]:
binner = KBinsDiscretizer(n_bins=4, encode='ordinal', strategy='quantile')
age_binned = binner.fit_transform(X_train_freq[['customer_age']])

pd.Series(age_binned.ravel()).value_counts().sort_index()

> **What just happened?** `strategy='quantile'` chose bin edges so each of the 4 bins has roughly the same number of customers. That's useful when you want balanced groups rather than fixed-width age ranges.

## Step 9: Assemble Everything into One Leakage-Safe Pipeline

Doing each transform by hand helped show what's happening, but it's error-prone and easy to leak. `ColumnTransformer` and `Pipeline` apply every transform in one `fit`/`transform` call, fit only on training data. The result can go straight into cross-validation or a SageMaker inference container.

In [ ]:
numeric_features = ['customer_age', 'items_in_cart', 'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos']
categorical_features = ['region', 'payment_method']

preprocessor = ColumnTransformer(transformers=[
    ('num', Pipeline([
        ('impute', SimpleImputer(strategy='median')),
        ('scale', StandardScaler()),
    ]), numeric_features),
    ('cat', Pipeline([
        ('impute', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore')),
    ]), categorical_features),
])

full_pipeline = Pipeline([
    ('preprocess', preprocessor),
    ('model', Ridge(alpha=1.0, random_state=RANDOM_SEED)),
])

# Rebuild train/test with the columns the pipeline expects (drop raw datetime + customer_id,
# keep engineered datetime columns from Step 3, reuse the frequency-encoded frame's columns)
X_train_final = X_train_freq[numeric_features + categorical_features]
X_test_final = X_test_freq[numeric_features + categorical_features]

full_pipeline.fit(X_train_final, y_train)
print(f"Test R^2: {full_pipeline.score(X_test_final, y_test):.3f}")

> **What just happened?** `ColumnTransformer` sent numeric columns through imputation and scaling, and categorical columns through imputation and one-hot encoding, all inside a single `Pipeline`. Calling `.fit()` fits every step on `X_train_final` only. `.score()` on the held-out test set gives an honest, leakage-free estimate of performance.

## Step 10: Cross-Validate the Full Pipeline

In [ ]:
scores = cross_val_score(full_pipeline, X_train_final, y_train, cv=5, scoring='r2')
print(f"CV R^2: {scores.mean():.3f} +/- {scores.std():.3f}")

> **What just happened?** Because `full_pipeline` includes the preprocessing steps, `cross_val_score` refits the scaler and encoder on each fold's training split. That's the correct way to cross-validate feature engineering. If you had scaled `X_train_final` once before calling `cross_val_score`, each fold's validation data would have leaked into the scaler's mean and standard deviation.

## Step 11: Leaky vs. CV-Safe Target Encoding

Step 5 used frequency encoding for `customer_id` because it's safe: how often a customer appears in training data can't leak the target. **Target encoding** replaces a category with the *mean target value* for that category. It's more informative when frequency alone isn't, but it's also the easiest place to leak the label by accident. The safe version encodes each row using only target statistics computed on *other* training folds, never the row's own fold. The leaky version computes the mean once, upfront, on the whole dataset (train and test together) before splitting.

In [ ]:
from sklearn.model_selection import KFold

def cv_target_encode(X_train, y_train, X_test, column, n_splits=5, smoothing=10):
    global_mean = y_train.mean()
    train_encoded = pd.Series(index=X_train.index, dtype=float)
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_SEED)
    for tr_idx, val_idx in kf.split(X_train):
        fold_train = X_train.iloc[tr_idx]
        fold_y = y_train.iloc[tr_idx]
        stats = fold_y.groupby(fold_train[column]).agg(['mean', 'count'])
        smoothed = (
            stats['mean'] * stats['count'] + global_mean * smoothing
        ) / (stats['count'] + smoothing)
        val_rows = X_train.iloc[val_idx][column]
        train_encoded.iloc[val_idx] = val_rows.map(smoothed).fillna(global_mean).values
    full_stats = y_train.groupby(X_train[column]).agg(['mean', 'count'])
    full_smoothed = (
        full_stats['mean'] * full_stats['count'] + global_mean * smoothing
    ) / (full_stats['count'] + smoothing)
    test_encoded = X_test[column].map(full_smoothed).fillna(global_mean)
    return train_encoded, test_encoded

# Leaky: mean target per customer computed on the FULL dataset (train + test),
# before the split ever happened -- test-set rows influence the training encoding
leaky_stats = df.groupby('customer_id')['order_value'].mean()
train_te_leaky = X_train['customer_id'].map(leaky_stats)
test_te_leaky = X_test['customer_id'].map(leaky_stats)

# Safe: Procedure 14.2's fold-holdout encoding, train-only throughout
train_te_safe, test_te_safe = cv_target_encode(X_train, y_train, X_test, 'customer_id')

X_train_leaky = X_train_dt.copy()
X_train_leaky['customer_te'] = train_te_leaky.values
X_test_leaky = X_test_dt.copy()
X_test_leaky['customer_te'] = test_te_leaky.values

X_train_safe = X_train_dt.copy()
X_train_safe['customer_te'] = train_te_safe.values
X_test_safe = X_test_dt.copy()
X_test_safe['customer_te'] = test_te_safe.values

feat_cols = numeric_features + categorical_features + ['customer_te']

te_preprocessor = ColumnTransformer(transformers=[
    ('num', Pipeline([
        ('impute', SimpleImputer(strategy='median')),
        ('scale', StandardScaler()),
    ]), numeric_features + ['customer_te']),
    ('cat', Pipeline([
        ('impute', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore')),
    ]), categorical_features),
])
ridge_te = Pipeline([
    ('preprocess', te_preprocessor),
    ('model', Ridge(alpha=1.0, random_state=RANDOM_SEED)),
])

scores_leaky = cross_val_score(ridge_te, X_train_leaky[feat_cols], y_train, cv=5, scoring='r2')
scores_safe = cross_val_score(ridge_te, X_train_safe[feat_cols], y_train, cv=5, scoring='r2')

print(f"Leaky encoding  CV R^2: {scores_leaky.mean():.3f} +/- {scores_leaky.std():.3f}")
print(f"Safe encoding   CV R^2: {scores_safe.mean():.3f} +/- {scores_safe.std():.3f}")

ridge_te.fit(X_train_leaky[feat_cols], y_train)
test_r2_leaky = ridge_te.score(X_test_leaky[feat_cols], y_test)
ridge_te.fit(X_train_safe[feat_cols], y_train)
test_r2_safe = ridge_te.score(X_test_safe[feat_cols], y_test)

print(f"Leaky encoding  test R^2: {test_r2_leaky:.3f}")
print(f"Safe encoding   test R^2: {test_r2_safe:.3f}")

> **What just happened?** The leaky encoding reports a slightly higher CV R² than the safe one (0.872 vs. 0.870), the direction the leakage warning predicts. The held-out test R² shows the same small gap (0.870 vs. 0.867), so it isn't a fluke. The gap is modest because `customer_id` isn't a strongly predictive column in this dataset. Spend is driven mostly by region, cart size, and weekend timing, not by which customer placed the order. That's a more precise lesson than "leakage always inflates your score a lot": the damage scales with how predictive the leaking column is. A weakly predictive ID leaks only a little. A strongly predictive one can produce a large, disqualifying gap. Measuring the gap, rather than assuming its size, is the only way to know which case you're in.

## What You Built

You turned a raw orders table (categoricals, a high-cardinality ID column, and a datetime) into a leakage-safe, cross-validation-ready feature matrix. You packaged the whole transformation into one reusable `Pipeline` that a Ridge regression can use end to end. And you measured, rather than assumed, how much a leaky target encoding costs.

## What's Next

- Read `explanation.md` to understand *why* each transformation works and when to use it.
- Open `howto.md` when you need to repeat one of these recipes on your own data.
- Keep `reference.md` open as a cheat sheet for encoder and scaler signatures.
- **Extension challenge:** rerun Step 11 with `smoothing=1` instead of `10` in `cv_target_encode`. Does a weaker smoothing prior widen or narrow the leaky-vs-safe gap, and why?

---

**AWS SageMaker version:** the same `ColumnTransformer` pipeline can be wrapped in a SageMaker Scikit-learn Processing job, or rebuilt as a SageMaker Data Wrangler flow for a no-code equivalent. See `howto.md` for the SageMaker Feature Store pattern.